In [1]:
# STEP 1: IMPORT MODULES
import os
import librosa
import numpy as np
import pandas as pd


In [2]:
# STEP 2: Set Dataset Path
DATA_PATH = r"C:\Users\Yogesh\Desktop\emotional_classification\emotion_project\data\RAVDESS"


In [3]:
# STEP 3: Emotion Label Extraction from Filename
def get_emotion_label(filename):
    emotion_code = int(filename.split("-")[2])
    emotion_map = {
        1: "neutral",
        2: "calm",
        3: "happy",
        4: "sad",
        5: "angry",
        6: "fearful",
        7: "disgust",
        8: "surprised"
    }
    return emotion_map[emotion_code]


In [4]:
# STEP 4: Feature Extraction Function
def extract_features(file_path):
    y, sr = librosa.load(file_path)

    # 1️⃣ MFCC (Spectral Features)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_mean = np.mean(mfcc, axis=1)

    # 2️⃣ PITCH (Prosodic Feature) — FIXED METHOD
    f0 = librosa.yin(y, fmin=50, fmax=300)
    pitch_mean = np.mean(f0[f0 > 0])

    # 3️⃣ ENERGY (Intensity Feature)
    energy = librosa.feature.rms(y=y)
    energy_mean = np.mean(energy)

    return mfcc_mean, pitch_mean, energy_mean


In [5]:
# STEP 5: Build Dataset from All Audio Files
data = []

for actor in os.listdir(DATA_PATH):
    actor_path = os.path.join(DATA_PATH, actor)
    
    for file in os.listdir(actor_path):
        if file.endswith(".wav"):
            file_path = os.path.join(actor_path, file)

            mfcc, pitch, energy = extract_features(file_path)
            emotion = get_emotion_label(file)

            row = list(mfcc) + [pitch, energy, emotion]
            data.append(row)


In [6]:
# STEP 6: Create DataFrame
columns = [f"mfcc_{i+1}" for i in range(13)] + ["pitch", "energy", "emotion"]
df = pd.DataFrame(data, columns=columns)

df.head()


,mfcc_1,mfcc_2,mfcc_3,mfcc_4,mfcc_5,mfcc_6,mfcc_7,mfcc_8,mfcc_9,mfcc_10,mfcc_11,mfcc_12,mfcc_13,pitch,energy,emotion
0,-697.792603,54.890041,0.663465,12.435786,7.733951,0.530750,-3.216631,-3.159395,-10.977551,-2.848711,0.815297,-3.037067,1.955447,88.506851,0.002256,neutral
1,-692.855774,55.363899,-1.548319,16.038305,8.818810,-0.146586,-1.373392,-5.293180,-11.623182,-1.348284,0.843715,-2.641278,1.017250,91.076464,0.002419,neutral
2,-691.587891,58.024662,0.159465,13.624650,5.374113,1.162336,-2.083359,-5.382585,-10.332824,-3.662081,0.560601,-2.838226,1.834785,90.451596,0.002809,neutral
3,-685.105469,55.879421,2.783262,13.252023,6.989669,2.981274,-1.586029,-6.961661,-10.348489,-3.270769,1.176165,-1.498656,0.551550,84.540114,0.002617,neutral
4,-727.104370,62.355034,3.121181,15.064671,8.132434,1.927084,-3.274656,-3.761792,-9.750299,-4.853837,0.996315,-3.116070,0.546321,86.179753,0.001653,calm


In [7]:
# STEP 7: Encode Emotion Labels
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
df["emotion_encoded"] = encoder.fit_transform(df["emotion"])

df[["emotion", "emotion_encoded"]].head()


,emotion,emotion_encoded
0,neutral,5
1,neutral,5
2,neutral,5
3,neutral,5
4,calm,1


In [8]:
# STEP 8: Train/Test Split & Scaling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


In [9]:
# STEP 9: MODEL A — MFCC ONLY
X_mfcc = df[[f"mfcc_{i+1}" for i in range(13)]]
y = df["emotion_encoded"]

X_train, X_test, y_train, y_test = train_test_split(
    X_mfcc, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model_mfcc = RandomForestClassifier(random_state=42)
model_mfcc.fit(X_train, y_train)

y_pred_mfcc = model_mfcc.predict(X_test)
acc_mfcc = accuracy_score(y_test, y_pred_mfcc)

print("MFCC-only Accuracy:", acc_mfcc)


MFCC-only Accuracy: 0.5868055555555556


In [10]:
# STEP 10: MODEL B — MFCC + Pitch + Energy
X_full = df[[f"mfcc_{i+1}" for i in range(13)] + ["pitch", "energy"]]

X_train, X_test, y_train, y_test = train_test_split(
    X_full, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model_full = RandomForestClassifier(random_state=42)
model_full.fit(X_train, y_train)

y_pred_full = model_full.predict(X_test)
acc_full = accuracy_score(y_test, y_pred_full)

print("MFCC + Pitch + Energy Accuracy:", acc_full)


MFCC + Pitch + Energy Accuracy: 0.5590277777777778


In [11]:
import joblib
import os

# Define the folder where you want to save them
save_path = r"C:\Users\Yogesh\Desktop\emotional_classification\emotion_project"

# Exporting the 'brains' of your project
joblib.dump(model_mfcc, os.path.join(save_path, "mfcc_model.pkl"))
joblib.dump(model_full, os.path.join(save_path, "mfcc_pitch_energy_model.pkl"))
joblib.dump(scaler, os.path.join(save_path, "scaler.pkl"))
joblib.dump(encoder, os.path.join(save_path, "encoder.pkl"))

print(f"✅ Success! 4 files saved in: {save_path}")

✅ Success! 4 files saved in: C:\Users\Yogesh\Desktop\emotional_classification\emotion_project
